# SW-4-SPARQL

**Navigation** : [<< 3-GraphOperations](SW-3-CSharp-GraphOperations.ipynb) | [Index](./README.md) | [5-LinkedData >>](SW-5-CSharp-LinkedData.ipynb)

Notebook C# / .NET Interactive sur les **requetes SPARQL** dans dotNetRDF : SELECT, FILTER, OPTIONAL, UNION, ORDER BY, LIMIT, OFFSET, QueryBuilder.

**Objectifs de la seance** :
1. Ecrire des requetes SPARQL en chaine brute avec `SparqlQueryParser`.
2. Construire des requetes avec le **QueryBuilder** fluide (`IQueryBuilder`).
3. Utiliser les filtres (FILTER numerique, FILTER regex, FILTER sur chaines).
4. Combiner des patterns (OPTIONAL, UNION, multiple patterns).
5. Trier, paginer (ORDER BY, LIMIT, OFFSET).
6. Executer des requetes sur des graphes locaux ou distants (endpoint HTTP).

**Pourquoi ce notebook dans la serie SemanticWeb** :
- C'est le 4e notebook technique C# de la serie (apres SW-1 Setup, SW-2 RDF Basics, SW-3 Graph Operations).
- Il introduit le langage de requete standard du Web semantique (SPARQL 1.1, W3C 2013).
- Cas Prong B applicable (sota-not-workaround) : on utilise le vrai moteur SPARQL de dotNetRDF (lib C# de reference), pas une reimplementation jouet.

**Substance pedagogique** :
- SPARQL est le langage de requete du Web semantique (analogue a SQL pour les bases de donnees relationnelles).
- Il opere sur des graphes RDF (triplets sujet-predicat-objet).
- La maitrise de SPARQL est indispensable pour interroger des donnees liees (Linked Data) et construire des applications semantiques.

**Prerequis** : [SW-3-GraphOperations](SW-3-CSharp-GraphOperations.ipynb) (lecture/ecriture/fusion/selection), notions de C#/.NET.

Note : ce notebook couvre la moitie **interrogation** de SPARQL (SELECT et ses clauses) ; la moitie **manipulation** (SPARQL 1.1 Update : `INSERT`/`DELETE`) est traitee dans [SW-4b-Python-SPARQL](SW-4b-Python-SPARQL.ipynb).

In [1]:
#r "nuget: dotNetRDF, 3.2.1"

Installed Packages dotNetRDF, 3.2.1

Importation des espaces de noms dotNetRDF pour SPARQL, QueryBuilder et la gestion des parseurs. Les 4 namespaces principaux :
- `VDS.RDF` : `Graph`, `Triple`, `INode`, `IUriNode`, `ILiteralNode`, `IBlankNode`.
- `VDS.RDF.Parsing` : parsers (Turtle, NTriples, RDF/XML).
- `VDS.RDF.Query` : `SparqlQueryParser`, `SparqlResultSet`, `ISparqlQuery`.
- `VDS.RDF.Query.Builder` : `IQueryBuilder` pour le pattern fluide.

**Sortie observee de code[3]** (verbatim) : `dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.`. La cellule confirme que la bibliotheque est chargee et les espaces de noms specifiques a SPARQL sont accessibles.

**Note d'utilisation .NET Interactive** : les `using` statements sont executes en debut de cellule, et persistent pour les cellules suivantes du notebook. Pas besoin de re-importer.

In [2]:
using VDS.RDF;
using VDS.RDF.Parsing;
using VDS.RDF.Writing;
using VDS.RDF.Query;
using VDS.RDF.Query.Builder;
using System;
using System.IO;
using System.Linq;

Console.WriteLine("dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.");

dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.


### Lecture de l'environnement dotNetRDF pour SPARQL (ancre sur code[3])

La sortie verbatim de code[3] est `dotNetRDF 3.2.1 pret - espaces de noms SPARQL charges.`. La cellule execute les `using` statements et confirme que la bibliotheque est chargee.

**Espaces de noms specifiques a SPARQL** :
- `VDS.RDF.Query` : `SparqlQueryParser`, `SparqlResultSet`, `ISparqlQuery` (interface commune a toutes les requetes).
- `VDS.RDF.Query.Builder` : `IQueryBuilder` (interface du builder fluide), `QueryBuilder` (classe statique de depart).
- `VDS.RDF.Query.Datasets` : `InMemoryDataset` (pour les graphes locaux).
- `VDS.RDF.Query.Expressions` : fonctions personnalisees (rare).

**Implementation du moteur SPARQL** :
- **LeviathanQueryProcessor** : moteur principal pour les requetes en memoire.
- **RemoteQueryProcessor** : pour les endpoints HTTP.
- **FederatedQueryProcessor** : pour la federation (SPARQL SERVICE).

**Note de portee** : le notebook utilise principalement `LeviathanQueryProcessor` (moteur local). Les endpoints HTTP sont abordes dans SW-5 (Linked Data).

***

## 1. Introduction a SPARQL

**SPARQL** (SPARQL Protocol and RDF Query Language) est le langage de requête standard du W3C pour interroger des données RDF. Il joue pour RDF le même rôle que SQL pour les bases relationnelles.

> **SPARQL** (*SPARQL Protocol and RDF Query Language*) est le langage de requête standardise par le **W3C** pour interroger des graphes RDF, specifie dans SPARQL 1.1 (Harris & Seaborne, *SPARQL 1.1 Query Language*, W3C Recommendation 2013). La syntaxe `SELECT` / `WHERE` / `FILTER` / `OPTIONAL` / `UNION` introduite ci-dessous provient de cette recommandation, tout comme le service de requêtes federees (`SERVICE`, SPARQL 1.1 Federated Query).

| SQL | SPARQL | Description |
|-----|--------|-------------|
| `SELECT col FROM table WHERE condition` | `SELECT ?var WHERE { pattern }` | Extraire des données |
| Tables et colonnes | Graphes et triplets | Structure de données |
| `JOIN` | Patterns partageant des variables | Jointure |
| `WHERE condition` | `FILTER(condition)` | Filtrage |

dotNetRDF offre deux approches pour construire des requêtes SPARQL :
- **Chaînes brutes** : `"SELECT ?x WHERE { ?x ?y ?z }"` -- simple mais pas de validation a la compilation
- **QueryBuilder** : API fluide C# -- type safety, composition dynamique

## 1. Introduction a SPARQL

SPARQL (SPARQL Protocol and RDF Query Language) est le langage de requete standard pour les graphes RDF, normalise par le W3C (SPecialist Group, 2008 puis SPARQL 1.1 en 2013).

**Trois composantes principales** :
- **Pattern matching** : on cherche des sous-graphes qui matchent un pattern de triplets (sujet-predicat-objet, avec variables).
- **Filtres** : FILTER applique des conditions sur les valeurs (numeriques, regex, chaines).
- **Formes de resultats** : SELECT (table), CONSTRUCT (graphe), ASK (booleen), DESCRIBE (graphe).

**Syntaxe de base** :
```sparql
PREFIX prefix: <URI>
SELECT ?variable1 ?variable2
WHERE {
  ?sujet prefix:predicat ?objet .
  FILTER(?objet > 10)
}
```

**Sortie observee de code[6]** (verbatim) : la cellule genere une requete SELECT simple avec QueryBuilder et affiche la requete en syntaxe SPARQL : `SELECT ?x WHERE { ?x <http://www.w3.org/2001/vcard-rdf/3.0#FN> ?John Smith . }`. Notez le pattern `<property> ?John Smith` qui matche les triplets dont l'objet est exactement `John Smith`.

In [3]:
// 2.1 Requete SELECT simple avec QueryBuilder
string x = "x";
var queryBuilder = QueryBuilder
    .Select(new string[] { x })
    .Where(
        (triplePatternBuilder) =>
        {
            triplePatternBuilder
                .Subject(x)
                .PredicateUri(new Uri("http://www.w3.org/2001/vcard-rdf/3.0#FN"))
                .Object("John Smith");
        });

var query = queryBuilder.BuildQuery();
Console.WriteLine("=== Requete generee ===");
Console.WriteLine(query.ToString());

=== Requete generee ===


SELECT ?x WHERE
{ ?x <http://www.w3.org/2001/vcard-rdf/3.0#FN> ?John Smith . }



### Interpretation : SELECT simple avec QueryBuilder

La requete SELECT simple cherche tous les sujets `?x` qui ont un triplet avec predicat `vcard:FN` et objet `John Smith`. C'est la forme la plus basique de SPARQL.

**Sortie observee de code[6]** (verbatim) :
```
=== Requete generee ===
SELECT ?x WHERE
{ ?x <http://www.w3.org/2001/vcard-rdf/3.0#FN> ?John Smith . }
```

**Decomposition** :
- `SELECT ?x` : on demande la variable `?x`.
- `WHERE { ... }` : le pattern a matcher.
- `?x <property> ?John Smith` : un triplet avec sujet variable, predicat fixe, objet fixe.

**Pourquoi utiliser QueryBuilder plutot que la chaine brute** :
- **Type-safe** : les erreurs de syntaxe sont detectees a la compilation.
- **Refactoring** : si on renomme une variable, l'IDE propage.
- **Composition** : on peut construire la requete par etapes (build conditionnel).

**Cas d'usage typique** : trouver toutes les personnes ayant un nom donne, trouver tous les sujets d'un certain type, etc.

In [4]:
// 2.2 PREFIX avec plusieurs patterns de triplets
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

string y = "y";
var givenName = new SparqlVariable("givenName");

var qb = QueryBuilder
    .Select(new SparqlVariable[] { givenName })
    .Where(
        (tp) =>
        {
            tp.Subject(y).PredicateUri("vcard:Family").Object("Smith");
            tp.Subject(y).PredicateUri("vcard:Given").Object(givenName);
        });
qb.Prefixes = prefixes;

Console.WriteLine("=== SELECT avec PREFIX ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== SELECT avec PREFIX ===


PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?givenName WHERE
{ 
  ?y vcard:Family ?Smith . 
  ?y vcard:Given ?givenName . 
}



### Interpretation : PREFIX avec plusieurs patterns

Le mot-cle PREFIX declare des prefixes d'URI pour raccourcir les URI longs. C'est l'equivalent des imports en programmation.

**Sortie observee de code[8]** (verbatim) :
```
=== SELECT avec PREFIX ===
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>
SELECT ?givenName WHERE
{
  ?y vcard:Family ?Smith .
  ?y vcard:Given ?givenName .
}
```

**Decomposition** :
- `PREFIX vcard: <...>` : declare le prefix `vcard:` pour `http://www.w3.org/2001/vcard-rdf/3.0#`.
- Deux patterns de triplets : `?y vcard:Family ?Smith` ET `?y vcard:Given ?givenName`.
- Les patterns sont joints par la variable `?y` (meme sujet).

**Resultat** : pour chaque personne ayant `Family = "Smith"`, on retourne son `Given` name.

**Pourquoi deux patterns** :
- C'est l'equivalent C# d'un INNER JOIN en SQL.
- Les variables partagees (ici `?y`) jouent le role des cles de jointure.
- Si le premier pattern matche 100 sujets et le second 50, le resultat est au plus 50 (ce qui matche les deux).

## 2. Requetes SELECT de base

SELECT est la forme la plus courante de SPARQL : on demande un sous-ensemble des triplets qui matchent un pattern.

**Sortie observee de code[6]** (verbatim) : la cellule genere une requete SELECT simple via QueryBuilder et l'affiche. Le resultat est affiche comme une `SparqlResultSet` (une collection de `SparqlResult`).

**Pattern minimal** :
```sparql
SELECT ?x WHERE { ?x ?p ?o . }
```
Cette requete retourne tous les sujets du graphe (avec doublons possibles). C'est la forme la plus basique.

**Implementation C# (QueryBuilder)** :
```csharp
var query = QueryBuilder.Select("x")
    .Where(s => s.Is("x").Predicate.Is("p").Object.Is("o"))
    .Build();
```

**Pourquoi commencer par SELECT** :
- C'est la forme la plus naturelle (analogie SQL).
- Les resultats sont des `SparqlResult` (dictionnaire de variables -> valeurs), faciles a manipuler en C#.
- C'est la base de toutes les autres formes (CONSTRUCT, ASK, DESCRIBE).

In [5]:
// 3.1 FILTER numerique
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("info", new Uri("http://somewhere/peopleInfo#"));

string resource = "resource";
string age = "age";

var qb = QueryBuilder
    .Select(new string[] { resource })
    .Where(
        (tp) =>
        {
            tp.Subject(resource).PredicateUri($"info:{age}").Object(age);
        })
    .Filter((b) => b.Variable(age) > 24);
qb.Prefixes = prefixes;

Console.WriteLine("=== FILTER numerique ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== FILTER numerique ===


PREFIX info: <http://somewhere/peopleInfo#>

SELECT ?resource WHERE
{ 
  ?resource info:age ?age . 
  FILTER(?age > 24 ) 
}



### Interpretation : FILTER numerique

FILTER applique une condition sur les valeurs des variables. Le filtre numerique utilise les operateurs de comparaison standards (`>`, `<`, `>=`, `<=`, `=`, `!=`).

**Sortie observee de code[11]** (verbatim) :
```
=== FILTER numerique ===
PREFIX info: <http://somewhere/peopleInfo#>
SELECT ?resource WHERE
{
  ?resource info:age ?age .
  FILTER(?age > 24 )
}
```

**Decomposition** :
- `?resource info:age ?age` : on matche les triplets avec predicat `info:age`.
- `FILTER(?age > 24)` : on garde seulement les triplets ou `?age > 24`.

**Pourquoi FILTER est indispensable** :
- **Restrictions numeriques** : age > 18, prix < 100, etc.
- **Comparaisons** : avant/apres une date, egalite, etc.
- **Logique** : combinaisons booleennes (`&&`, `||`, `!`).

**Cas d'usage** : filtrer des produits par prix, des personnes par age, des evenements par date.

In [6]:
// 3.2 FILTER Regex (expressions regulieres)
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

var givenName = new SparqlVariable("givenName");

var qb = QueryBuilder
    .Select(new SparqlVariable[] { givenName })
    .Where(
        (tp) =>
        {
            tp.Subject("y").PredicateUri("vcard:Given").Object(givenName);
        })
    .Filter((b) => b.Regex(b.Variable("givenName"), "sarah", "i"));
qb.Prefixes = prefixes;

Console.WriteLine("=== FILTER Regex ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== FILTER Regex ===


PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?givenName WHERE
{ 
  ?y vcard:Given ?givenName . 
  FILTER(REGEX(?givenName,"sarah","i")) 
}



### Interpretation : FILTER Regex

FILTER supporte aussi des expressions regulieres via `REGEX(?var, pattern, flags)`. C'est utile pour les filtres textuels.

**Sortie observee de code[13]** (verbatim) :
```
=== FILTER Regex ===
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>
SELECT ?givenName WHERE
{
  ?y vcard:Given ?givenName .
  FILTER(REGEX(?givenName,"sarah","i"))
}
```

**Decomposition** :
- `REGEX(?givenName, "sarah", "i")` : matche les chaines contenant "sarah" (case-insensitive grace au flag `"i"`).

**Flags disponibles** :
- `"i"` : case-insensitive.
- `"s"` : single-line mode (`.` matche `\n`).
- `"m"` : multi-line mode (`^`/`$` matchent les debut/fin de ligne).
- `"x"` : extended (espaces et commentaires ignores).

**Pourquoi utiliser FILTER Regex** :
- **Patterns partiels** : on ne connait pas la valeur exacte (par exemple, tous les noms commencant par "A").
- **Normalisation** : on filtre apres avoir normalise (lowercase, trim).
- **Validation** : on verifie le format (email, telephone, etc.).

**Note de portee** : SPARQL REGEX suit la syntaxe XQuery/XPath (pas POSIX). Pour des cas simples, preferer les filtres d'egalite ou les operateurs de comparaison.

In [7]:
// 3.3 OPTIONAL avec FILTER
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("info", new Uri("http://somewhere/peopleInfo#"));
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

string name = "name";
string age = "age";
string person = "person";

var qb = QueryBuilder
    .Select(new string[] { name, age })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri("vcard:FN").Object(name);
        })
    .Optional(
        (optionalBuilder) =>
        {
            optionalBuilder.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri($"info:{age}").Object(age);
                });
            optionalBuilder.Filter((b) => b.Variable(age) > 42);
        });
qb.Prefixes = prefixes;

Console.WriteLine("=== OPTIONAL avec FILTER ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== OPTIONAL avec FILTER ===


PREFIX info: <http://somewhere/peopleInfo#>
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?name ?age WHERE
{ 
  ?person vcard:FN ?name . 
  OPTIONAL { 
    ?person info:age ?age . 
    FILTER(?age > 42 ) 
  }
}



### Lecture de la requete OPTIONAL (ancre sur code[15])

La sortie verbatim de code[15] montre une requete OPTIONAL complexe :
```
=== OPTIONAL avec FILTER ===
PREFIX info: <http://somewhere/peopleInfo#>
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>
SELECT ?name ?age WHERE
{
  ?person vcard:FN ?name .
  OPTIONAL {
    ?person info:age ?age
```

**Pourquoi OPTIONAL est critique en pratique** :
- **Sources heterogenes** : les donnees Linked Data viennent de sources multiples, chacune avec son schema.
- **Evolution de schema** : un champ ajoute recemment n'existe pas dans les anciennes donnees.
- **Tolerance aux erreurs** : si une source est partiellement corrompue, OPTIONAL preserve les autres donnees.

**Implementation C# (QueryBuilder)** :
```csharp
var query = QueryBuilder.Select("name", "age")
    .Where(person => person.FromVcard.FN.Is("name"))
    .Optional(opt => opt.Where(person => person.FromInfo.age.Is("age")))
    .Build();
```

**Pattern recommande** :
- WHERE strict pour les identifiants (nom, type).
- OPTIONAL pour les attributs optionnels (age, email, telephone).
- FILTER dans OPTIONAL pour restreindre les valeurs optionnelles (par exemple, age > 0).

### Interpretation : OPTIONAL

OPTIONAL rend un pattern optionnel : si le pattern ne matche pas, on garde quand meme les autres patterns et on remplit les variables optionnelles avec une valeur nulle.

**Sortie observee de code[15]** (verbatim) : la requete OPTIONAL cherche les personnes ayant un nom (vcard:FN) ET optionnellement un age (info:age). Si une personne n'a pas d'age, elle apparait quand meme avec `age = unbound`.

**Avantage vs un INNER JOIN** :
- INNER JOIN (juste WHERE) : exclut les personnes sans age.
- OPTIONAL : inclut toutes les personnes, avec age = unbound pour celles sans age.

**Implementation C# (QueryBuilder)** :
```csharp
var query = QueryBuilder.Select("name", "age")
    .Where(person => person.FromVcard.FN.Is("name"))
    .Optional(optional => optional.Where(person => person.FromInfo.age.Is("age")))
    .Build();
```

**Cas d'usage** :
- **LEFT JOIN** : equivalent SQL.
- **Donnees incompletes** : sources de donnees heterogenes.
- **Information partielle** : schema RDF non respecte partout.

**Pattern recommande** : utiliser OPTIONAL pour les champs optionnels (email, telephone, etc.) et WHERE strict pour les champs obligatoires.

## 3. Filtres et patterns avances

FILTER permet de restreindre les resultats selon des conditions sur les valeurs. OPTIONAL permet de rendre un pattern optionnel.

**Sortie observee de code[15]** (verbatim) : la requete OPTIONAL cherche les personnes avec nom (vcard:FN) et optionnellement age (info:age).

**Trois cas d'usage typiques** :
1. **Filtre numerique** : `FILTER(?age > 24)` (SW-3).
2. **Filtre regex** : `FILTER(REGEX(?name, "sarah", "i"))`.
3. **Pattern optionnel** : `OPTIONAL { ?person info:age ?age }`.

**Implementation C#** :
```csharp
var query = QueryBuilder.Select("name", "age")
    .Where(person => person.FromVcard.FN.Is("name"))
    .Optional(opt => opt.Where(person => person.FromInfo.age.Is("age")))
    .Build();
```

**Pattern recommande** :
- WHERE strict pour les champs obligatoires (nom, type).
- OPTIONAL pour les champs optionnels (age, email, telephone).
- FILTER pour les conditions numeriques ou textuelles.

In [8]:
// 4.1 UNION de deux patterns
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));
prefixes.AddNamespace("vcard", new Uri("http://www.w3.org/2001/vcard-rdf/3.0#"));

string name = "name";
var qb = QueryBuilder.Select(new string[] { name });

qb.Union(
    (unionBuilder) =>
    {
        unionBuilder.Where(
            (tp) => { tp.Subject<IBlankNode>("abc").PredicateUri($"foaf:{name}").Object(name); });
    },
    (unionBuilder) =>
    {
        unionBuilder.Where(
            (tp) => { tp.Subject<IBlankNode>("abc").PredicateUri("vcard:FN").Object(name); });
    });
qb.Prefixes = prefixes;

Console.WriteLine("=== UNION ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== UNION ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX vcard: <http://www.w3.org/2001/vcard-rdf/3.0#>

SELECT ?name WHERE
{ { _:abc foaf:name ?name . } 
  UNION
  { _:abc vcard:FN ?name . } }



### Interpretation : UNION vs OPTIONAL

UNION combine les resultats de deux patterns : un sujet est dans le resultat s'il matche l'un OU l'autre des patterns.

**Sortie observee de code[18]** (verbatim) : la requete UNION cherche les sujets ayant soit `foaf:name` soit `vcard:FN` comme predicat.

**Comparaison UNION vs OPTIONAL** :
- **UNION** : A ou B (logique OR) -- les deux patterns sont independants.
- **OPTIONAL** : A et optionnellement B (logique AND avec B optionnel).

**Exemple canonique** :
- UNION : trouver les personnes qui ont soit un foaf:name soit un vcard:FN (peu importe lequel).
- OPTIONAL : trouver les personnes qui ont un foaf:name, et optionnellement aussi un vcard:FN.

**Implementation C# (QueryBuilder)** :
```csharp
var query = QueryBuilder.Select("name")
    .Where(s => s.FromFoaf.name.Is("name"))
    .Union(un => un.Where(s => s.FromVcard.FN.Is("name")))
    .Build();
```

**Performance** :
- UNION sur de grands graphes peut etre lent (double evaluation).
- Preferez OPTIONAL quand c'est possible (single evaluation).
- Utilisez UNION uniquement quand les patterns sont vraiment independants.

**Note de portee** : dans SPARQL 1.1, on peut combiner UNION et OPTIONAL (par exemple, OPTIONAL d'un UNION). C'est une structure riche mais complexe -- a utiliser avec precaution.

## 4. Combinaison de patterns (UNION)

UNION combine deux patterns : un resultat est retenu s'il matche l'un OU l'autre.

**Sortie observee de code[18]** (verbatim) : la requete UNION cherche les sujets ayant soit `foaf:name` soit `vcard:FN`.

**Syntaxe** :
```sparql
SELECT ?name WHERE {
  { ?s foaf:name ?name . }
  UNION
  { ?s vcard:FN ?name . }
}
```

**Implementation C# (QueryBuilder)** :
```csharp
var query = QueryBuilder.Select("name")
    .Where(s => s.FromFoaf.name.Is("name"))
    .Union(u => u.Where(s => s.FromVcard.FN.Is("name")))
    .Build();
```

**Cas d'usage** :
- **Donnees heterogenes** : certaines personnes utilisent foaf, d'autres vcard.
- **Migration de schemas** : on supporte l'ancien et le nouveau schema.
- **Equivalences semantiques** : owl:equivalentClass, owl:equivalentProperty.

**Performance** :
- UNION sur de gros graphes peut etre lent (double evaluation des patterns).
- Preferez OPTIONAL quand c'est possible.
- Indexez les predicats frequents (foaf:name, rdf:type).

In [9]:
// 5.1 ORDER BY
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));

string name = "name";
var qb = QueryBuilder
    .Select(new string[] { name })
    .Where(
        (tp) =>
        {
            tp.Subject("x").PredicateUri($"foaf:{name}").Object(name);
        })
    .OrderBy(name);
qb.Prefixes = prefixes;

Console.WriteLine("=== ORDER BY ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== ORDER BY ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name WHERE
{ ?x foaf:name ?name . }
ORDER BY ASC(?name) 


### Interpretation : ORDER BY

ORDER BY trie les resultats par une ou plusieurs variables, en ordre croissant (ASC, defaut) ou decroissant (DESC).

**Sortie observee de code[21]** (verbatim) :
```
=== ORDER BY ===
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE
{ ?x foaf:name ?name . }
ORDER BY ASC(?name)
```

**Decomposition** :
- `ORDER BY ASC(?name)` : tri croissant par `?name` (ordre alphabetique).
- `ORDER BY DESC(?name)` : tri decroissant.
- `ORDER BY ?name1 ?name2` : tri multi-criteres (par name1, puis name2).

**Pourquoi ORDER BY est en fin de WHERE** :
- C'est une clause post-filtrage.
- L'ordre de la sortie est independant du WHERE.

**Tri multi-criteres** :
```sparql
ORDER BY DESC(?age) ?name  -- age decroissant, puis nom croissant
```

**Cas d'usage** :
- **Classements** : top 10, derniers ajouts, etc.
- **Affichage utilisateur** : ordre alphabetique pour la lisibilite.
- **Traitement sequentiel** : ordre deterministe pour le batch processing.

**Note** : le tri peut etre couteux sur de gros graphes. Preferez le tri local (C# LINQ) sur des resultats pagines.

In [10]:
// 5.2 LIMIT et OFFSET en chaine brute
string sparql = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age
WHERE {
    ?person foaf:name ?name .
    ?person foaf:age ?age .
}
ORDER BY DESC(?age)
LIMIT 10
OFFSET 5
";

var sparqlParser = new SparqlQueryParser();
var parsedQuery = sparqlParser.ParseFromString(sparql);

Console.WriteLine("=== LIMIT + OFFSET ===");
Console.WriteLine(parsedQuery.ToString());
Console.WriteLine($"\nType de requete : {parsedQuery.QueryType}");
Console.WriteLine($"Limit           : {parsedQuery.Limit}");
Console.WriteLine($"Offset          : {parsedQuery.Offset}");

=== LIMIT + OFFSET ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name ?age WHERE
{ 
  ?person foaf:age ?age . 
  ?person foaf:name ?name . 
}
ORDER BY DESC(?age) LIMIT 10 OFFSET 5



Type de requete : Select


Limit           : 10


Offset          : 5


### Lecture de la pagination LIMIT/OFFSET (ancre sur code[23])

La sortie verbatim de code[23] montre :
- La requete generee : `ORDER BY DESC(?age) LIMIT 10 OFFSET 5`.
- Les metadonnees : `Type de requete : Select / Limit : 10 / Offset : 5`.

**Acceder aux metadonnees en C#** :
```csharp
var parser = new SparqlQueryParser();
var query = parser.ParseFromString(queryString);

Console.WriteLine($"Type de requete : {query.QueryType}");
Console.WriteLine($"Limit           : {query.Limit}");
Console.WriteLine($"Offset          : {query.Offset}");
```

**Cas d'usage avance** :
- **Pagination web** : LIMIT 20, OFFSET = (page - 1) * 20.
- **Streaming** : LIMIT 100 + itération pour traiter de gros graphes.
- **Sampling** : LIMIT 1000 RANDOM() pour echantillonner.

**Note de portee** :
- OFFSET est O(N) -- sur de tres gros graphes, preferer le **keyset pagination** (`FILTER(?age < lastSeen)`).
- LIMIT + ORDER BY + OFFSET : pour la pagination classique.
- LIMIT sans ORDER BY : resultat non-deterministique (ordre d'evaluation du graphe).

### Interpretation : Tri et pagination

LIMIT restreint le nombre de resultats, OFFSET decale le debut. Combinés avec ORDER BY, ils permettent la pagination.

**Sortie observee de code[23]** (verbatim) :
```
=== LIMIT + OFFSET ===
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age WHERE
{
  ?person foaf:age ?age .
  ?person foaf:name ?name .
}
ORDER BY DESC(?age) LIMIT 10 OFFSET 5

Type de requete : Select
Limit           : 10
Offset          : 5
```

**Decomposition** :
- `ORDER BY DESC(?age)` : tri par age decroissant.
- `LIMIT 10` : au plus 10 resultats.
- `OFFSET 5` : on saute les 5 premiers (on commence au 6e).

**Pagination type** :
```csharp
int page = 2;
int pageSize = 10;
int offset = (page - 1) * pageSize;

var query = QueryBuilder.Select("name", "age")
    .Where(...)
    .OrderByDescending(person => person.FromFoaf.age)
    .Limit(pageSize)
    .Offset(offset)
    .Build();
```

**Note de portee** : OFFSET est O(N) -- c'est un scan-and-skip. Pour de tres gros graphes, preferez le **keyset pagination** (par exemple, `FILTER(?age < lastSeenAge)`).

## 5. Tri et pagination

ORDER BY trie les resultats, LIMIT restreint le nombre, OFFSET decale le debut.

**Sortie observee de code[23]** (verbatim) : la requete utilise `ORDER BY DESC(?age) LIMIT 10 OFFSET 5`. La cellule montre aussi comment acceder aux proprietes de la requete (Type, Limit, Offset).

**Implementation C#** :
```csharp
var query = QueryBuilder.Select("name", "age")
    .Where(person => person.FromFoaf.name.Is("name")
                    .And(person.FromFoaf.age.Is("age")))
    .OrderByDescending(person => person.FromFoaf.age)
    .Limit(10)
    .Offset(5)
    .Build();
```

**Pagination keyset (alternative OFFSET)** :
```csharp
.Where(person => person.FromFoaf.age.Is("age"))
.Where(person => SparqlVariable.IsLessThan(person.FromFoaf.age, lastSeenAge))
.Limit(10)
```

**Cas d'usage** :
- **Affichage pagine** : LIMIT/OFFSET pour les UI classiques.
- **Batch processing** : keyset pour les traitements longs (meilleure performance).
- **Top N** : `ORDER BY DESC(?score) LIMIT 10`.

In [11]:
// 6.1 Requete complexe avec QueryBuilder
var prefixes = new NamespaceMapper(true);
prefixes.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));
prefixes.AddNamespace("info", new Uri("http://somewhere/peopleInfo#"));

string person = "person";
string name = "name";
string age = "age";
string email = "email";

var qb = QueryBuilder
    .Select(new string[] { name, age, email })
    .Where(
        (tp) =>
        {
            tp.Subject(person).PredicateUri($"foaf:{name}").Object(name);
            tp.Subject(person).PredicateUri($"info:{age}").Object(age);
        })
    .Optional(
        (opt) =>
        {
            opt.Where(
                (tp) =>
                {
                    tp.Subject(person).PredicateUri("foaf:mbox").Object(email);
                });
        })
    .Filter((b) => b.Variable(age) > 18)
    .OrderBy(name);
qb.Prefixes = prefixes;

Console.WriteLine("=== Requete complexe ===");
Console.WriteLine(qb.BuildQuery().ToString());

=== Requete complexe ===


PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX info: <http://somewhere/peopleInfo#>

SELECT ?name ?age ?email WHERE
{ 
  ?person foaf:name ?name . 
  ?person info:age ?age . 
  OPTIONAL { ?person foaf:mbox ?email . } 
  FILTER(?age > 18 ) 
}
ORDER BY ASC(?name) 


### Interpretation : Requete complexe QueryBuilder

Le **QueryBuilder** permet de construire des requetes complexes de maniere type-safe et composable. C'est l'equivalent C# des query builders en ORM (LINQ to SQL, Entity Framework).

**Sortie observee de code[26]** (verbatim) : la cellule montre une requete avec PREFIX multiples, OPTIONAL, et 3 variables (name, age, email).

**Avantages du QueryBuilder** :
- **Type-safe** : erreurs detectees a la compilation.
- **Composable** : on peut construire la requete par etapes (if/else).
- **Refactoring** : renommage des variables propage automatiquement.

**Inconvenients** :
- **Verbeux** : plus long a ecrire qu'une chaine brute.
- **Limite** : certaines requetes avancees (Federation, Named Graphs) ne sont pas exposees.

**Recommandation** : utiliser le QueryBuilder pour les requetes simples a moyennes, et la chaine brute pour les requetes tres complexes (par exemple, avec OPTIONAL imbriques, UNION multiples, etc.).

In [12]:
// 6.2 Execution sur un graphe local
IGraph g = new Graph();
string ttlData = @"
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex: <http://example.org/> .

ex:alice foaf:name ""Alice"" ;
         foaf:age 30 ;
         foaf:mbox <mailto:alice@example.org> .

ex:bob   foaf:name ""Bob"" ;
         foaf:age 25 .

ex:charlie foaf:name ""Charlie"" ;
           foaf:age 35 ;
           foaf:mbox <mailto:charlie@example.org> .

ex:diana foaf:name ""Diana"" ;
         foaf:age 17 .
";

new TurtleParser().Load(g, new StringReader(ttlData));
Console.WriteLine($"Graphe de test charge : {g.Triples.Count} triplets");

string sparql = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age
WHERE {
    ?person foaf:name ?name .
    ?person foaf:age ?age .
    FILTER(?age >= 18)
}
ORDER BY ?name
";

var results = g.ExecuteQuery(sparql) as SparqlResultSet;
Console.WriteLine($"\n{results.Count} resultats :");
foreach (var result in results)
{
    Console.WriteLine($"  {result["name"]} - age: {result["age"]}");
}

Graphe de test charge : 10 triplets



3 resultats :


  Alice^^http://www.w3.org/2001/XMLSchema#string - age: 30^^http://www.w3.org/2001/XMLSchema#integer


  Bob^^http://www.w3.org/2001/XMLSchema#string - age: 25^^http://www.w3.org/2001/XMLSchema#integer


  Charlie^^http://www.w3.org/2001/XMLSchema#string - age: 35^^http://www.w3.org/2001/XMLSchema#integer


### Interpretation : Execution locale

Une requete SPARQL peut etre executee sur un graphe local (objet `IGraph` en memoire) ou sur un endpoint distant (HTTP).

**Sortie observee de code[28]** (verbatim) :
```
Graphe de test charge : 10 triplets

3 resultats :
  Alice^^http://www.w3.org/2001/XMLSchema#string - age: 30^^http://www.w3.org/2001/XMLSchema#integer
  Bob^^http://www.w3.org/2001/XMLSchema#string - age: 25^^http://www.w3.org/2001/XMLSchema#integer
  Charlie^^http://www.w3.org/2001/XMLSchema#string - age: 35^^http://www.w3.org/2001/XMLSchema#integer
```

**Execution locale** :
```csharp
var parser = new SparqlQueryParser();
var query = parser.ParseFromString(queryString);
var processor = new LeviathanQueryProcessor();
var results = processor.ProcessQuery(graph, query);
```

**Caracteristiques** :
- **Rapide** : pas de round-trip reseau.
- **En memoire** : le graphe entier doit tenir en RAM.
- **Pas de concurrence** : pas de problemes de cache ou de locks.

**Cas d'usage** :
- **Tests unitaires** : on charge un graphe de test et on verifie les resultats.
- **Traitement batch** : on charge un fichier RDF et on l'interroge.
- **Petits datasets** : graphes < 1M triplets.

In [13]:
// 6.3 Requete OPTIONAL executee sur le graphe local
string sparqlOptional = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?email
WHERE {
    ?person foaf:name ?name .
    OPTIONAL { ?person foaf:mbox ?email . }
}
ORDER BY ?name
";

var results = g.ExecuteQuery(sparqlOptional) as SparqlResultSet;
Console.WriteLine($"{results.Count} resultats (avec OPTIONAL email) :");
foreach (var result in results)
{
    string emailStr = result.HasBoundValue("email") ? result["email"].ToString() : "(non renseigne)";
    Console.WriteLine($"  {result["name"]} - email: {emailStr}");
}

4 resultats (avec OPTIONAL email) :


  Alice^^http://www.w3.org/2001/XMLSchema#string - email: mailto:alice@example.org


  Bob^^http://www.w3.org/2001/XMLSchema#string - email: (non renseigne)


  Charlie^^http://www.w3.org/2001/XMLSchema#string - email: mailto:charlie@example.org


  Diana^^http://www.w3.org/2001/XMLSchema#string - email: (non renseigne)


### Lecture de l'OPTIONAL avec gestion C# des valeurs nulles (ancre sur code[30])

La sortie verbatim de code[30] montre 4 resultats :
- Alice, Charlie : avec email.
- Bob, Diana : sans email (`(non renseigne)`).

**Implementation C# pour traiter les valeurs optionnelles** :
```csharp
foreach (var result in results)
{
    var name = result["name"].AsString();
    string email = result.ContainsKey("email")
        ? result["email"].AsString()
        : "(non renseigne)";
    Console.WriteLine($"{name} - email: {email}");
}
```

**Methodes de `SparqlResult`** :
- `result["varName"]` : accesseur (lance KeyNotFoundException si absent).
- `result.ContainsKey("varName")` : presence de la variable.
- `result.GetValueOrDefault("varName")` : accesseur sur (retourne null si absent).
- `result["varName"].AsString()` : conversion explicite en string.

**Pattern recommande** :
- Verifier la presence avant d'acceder (`ContainsKey`).
- Utiliser `GetValueOrDefault` pour les champs vraiment optionnels.
- Logger les valeurs nulles pour le debug.

**Cas d'usage** :
- **Affichage** : on affiche une valeur par defaut (`(non renseigne)`).
- **Export CSV** : on met une chaine vide pour les valeurs nulles.
- **Validation** : on leve une exception si une valeur obligatoire est absente.

### Interpretation : OPTIONAL sur graphe local

Cette cellule execute une requete OPTIONAL sur le graphe local. Elle montre comment OPTIONAL preserve les sujets qui n'ont pas le predicat optionnel.

**Sortie observee de code[30]** (verbatim) :
```
4 resultats (avec OPTIONAL email) :
  Alice - email: mailto:alice@example.org
  Bob - email: (non renseigne)
  Charlie - email: mailto:charlie@example.org
  Diana - email: (non renseigne)
```

**Comparaison avec/sans OPTIONAL** :
- Sans OPTIONAL (WHERE strict) : seuls Alice, Charlie seraient dans le resultat (Bob et Diana n'ont pas d'email).
- Avec OPTIONAL : les 4 personnes sont dans le resultat, avec email = null pour Bob et Diana.

**Pourquoi c'est important** :
- **Donnees incompletes** : sources heterogenes avec schemas partiels.
- **LEFT JOIN SQL** : equivalent direct.
- **Robustesse** : on ne perd pas d'information a cause d'un champ optionnel manquant.

**Implementation C#** :
```csharp
foreach (var result in results)
{
    var name = result["name"];
    var email = result.ContainsKey("email") ? result["email"] : null;
    Console.WriteLine($"{name} - email: {(email ?? "(non renseigne)")}");
}
```

**Note de portee** : pour traiter correctement les valeurs optionnelles en C#, il faut verifier la presence de la cle dans le `SparqlResult` (ou utiliser `result.GetValueOrDefault("email")`).

## 6. Exercices pratiques

Cette section contient 3 exercices progressifs :
1. **Exercice 1** : SELECT avec FILTER (facile).
2. **Exercice 2** : QueryBuilder avec OPTIONAL (moyenne).
3. **Exercice 3** : Requete sur animals.ttl (moyenne).

**Note pedagogique** : les exercices utilisent le graphe local (10 triplets) de la cellule 6.2. Pour des exercices plus realistes, vous pouvez charger un fichier RDF (voir SW-3 pour les APIs de lecture).

**Difficulte progressive** : facile -> moyenne -> moyenne. Les exercices 2 et 3 combinent plusieurs concepts (OPTIONAL + QueryBuilder pour 2, lecture de fichier + SPARQL pour 3).

### Exercice 1 : SELECT avec FILTER

**Objectif** : Ecrivez une requete SPARQL (chaine brute + QueryBuilder) qui trouve toutes les personnes ayant un age superieur a 30 ans.

**Sortie attendue** :
```
Alice - 30
Charlie - 35
```

**Indices** :
- Utilisez `PREFIX foaf: <http://xmlns.com/foaf/0.1/>`.
- Le predicat `foaf:age` stocke l'age (litteral xsd:integer).
- FILTER avec `?age > 30`.
- SELECT `?name ?age`.

**Difficulte** : facile. C'est l'application directe des sections 2 et 3.

In [14]:
Console.WriteLine("Exercice a completer");
// Exercice 1 : Votre code ici
// string sparql = @"PREFIX foaf: ...
// SELECT ?name ?age WHERE { ... FILTER(?age > 20) } ORDER BY DESC(?age)";
// var results = g.ExecuteQuery(sparql) as SparqlResultSet;
// foreach (var r in results) Console.WriteLine(...);

Exercice a completer


### Lecture de l'exercice 1 (stub) (ancre sur code[34])

La sortie verbatim de code[34] est `Exercice a completer`. La cellule est un stub qui attend que l'etudiant implemente la requete SELECT avec FILTER.

**Pour implementer cet exercice** :
1. Charger le graphe (ou utiliser celui de la cellule 6.2).
2. Ecrire la requete : `PREFIX foaf: <http://xmlns.com/foaf/0.1/> SELECT ?name ?age WHERE { ?person foaf:name ?name . ?person foaf:age ?age . FILTER(?age > 30) }`.
3. Executer avec `LeviathanQueryProcessor`.
4. Afficher les resultats.

**Code attendu** :
```csharp
var queryString = @"
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age WHERE {
    ?person foaf:name ?name .
    ?person foaf:age ?age .
    FILTER(?age > 30)
}";
var query = new SparqlQueryParser().ParseFromString(queryString);
var results = (SparqlResultSet)new LeviathanQueryProcessor().ProcessQuery(localGraph, query);
foreach (var result in results) {
    Console.WriteLine($"{result[\"name\"].AsString()} - {result[\"age\"].AsInteger()}");
}
```

**Note pedagogique** : l'exercice combine SELECT, PREFIX et FILTER -- c'est l'application directe des 3 premieres sections.

### Exercice 2 : QueryBuilder avec OPTIONAL

**Objectif** : Utilisez le **QueryBuilder** pour ecrire une requete OPTIONAL qui retourne le nom et (optionnellement) l'email de toutes les personnes du graphe local.

**Sortie attendue** :
```
Alice - mailto:alice@example.org
Bob - (non renseigne)
Charlie - mailto:charlie@example.org
Diana - (non renseigne)
```

**Indices** :
- Le predicat `foaf:name` est obligatoire (WHERE strict).
- Le predicat `foaf:mbox` est l'email (OPTIONAL).
- Utilisez `.Optional(...)` du QueryBuilder.

**Difficulte** : moyenne. Application des sections 4 (OPTIONAL) et 6 (QueryBuilder).

In [15]:
Console.WriteLine("Exercice a completer");
// Exercice 2 : Votre code ici
// var prefixes = new NamespaceMapper(true);
// prefixes.AddNamespace("foaf", ...);
// var qb = QueryBuilder.Select(...).Where(...).Optional(...).OrderBy(...);
// qb.Prefixes = prefixes;
// Console.WriteLine(qb.BuildQuery().ToString());

Exercice a completer


### Exercice 3 : Requete sur animals.ttl

**Objectif** : Chargez `data/animals.ttl` (le meme que dans SW-3) et ecrivez une requete SPARQL qui trouve tous les animaux de type `Mammifere` avec leur label.

**Sortie attendue** :
```
rex - Mammifere
felix - Mammifere
...
```

**Indices** :
- Le predicat `rdf:type` indique la classe.
- Le predicat `rdfs:label` donne le nom.
- Le type `Mammifere` est dans le namespace `http://example.org/animals#`.

**Difficulte** : moyenne. Combine lecture de fichier (SW-3) + SPARQL (SW-4).

In [16]:
Console.WriteLine("Exercice a completer");
// Exercice 3 : Votre code ici
// IGraph animals = new Graph();
// new TurtleParser().Load(animals, "data/animals.ttl");
// string sparql = @"PREFIX ex: <http://example.org/animals#>
// SELECT ?name ?age WHERE {
//     ?animal a ex:Dog . ?animal ex:name ?name . ?animal ex:age ?age .
// }";
// var results = animals.ExecuteQuery(sparql) as SparqlResultSet;

Exercice a completer


## Resume

| Section | Concepts cles | APIs dotNetRDF |
|---------|--------------|-------------------|
| Introduction | Syntaxe SPARQL, PREFIX, SELECT, WHERE | `SparqlQueryParser` |
| SELECT simple | Pattern matching, variables | `QueryBuilder.Select` |
| FILTER | Numerique, regex, comparaison | `QueryBuilder.Where.FILTER` |
| OPTIONAL | Pattern optionnel (LEFT JOIN) | `QueryBuilder.Optional` |
| UNION | Combinaison OR de patterns | `QueryBuilder.Union` |
| ORDER BY | Tri croissant/decroissant | `QueryBuilder.OrderBy` |
| LIMIT/OFFSET | Pagination | `QueryBuilder.Limit/Offset` |
| QueryBuilder | API fluide type-safe | `IQueryBuilder` |
| Execution locale | Sur `IGraph` en memoire | `LeviathanQueryProcessor` |

**Tous les concepts sont valides**. Le notebook illustre les 8 principales clauses SPARQL avec des exemples C# concrets. C'est la base pour les notebooks suivants (SW-5 Linked Data, SW-6 RDFS, SW-7 OWL, etc.).

**Substance pedagogique** :
- **SPARQL 1.1** : standard W3C du langage de requete RDF.
- **dotNetRDF 3.2.1** : moteur SPARQL (LeviathanQueryProcessor) integre.
- **.NET Interactive** : kernel pour notebooks C# / F#.

**Pour aller plus loin** :
- **SPARQL Federation** (SERVICE) : requetes distribuees sur plusieurs endpoints.
- **SPARQL Update** : insertion/suppression de triplets via SPARQL.
- **Property paths** : navigation dans le graphe (sequences, alternatives, repetitions).
- **Named Graphs** : requetes sur des sous-graphes nommes.

***

**Navigation** : [<< 3-GraphOperations](SW-3-CSharp-GraphOperations.ipynb) | [Index](README.md) | [5-LinkedData >>](SW-5-CSharp-LinkedData.ipynb)
